Sara Rezaei

Alireza Nejati

# Task Description

![image.png](../assets/fig-01.png)

# Import Libraries

In [ ]:
import torch
import os
from PIL import Image
from transformers import CLIPProcessor, CLIPModel, AutoProcessor, AutoModelForVision2Seq ,  BlipProcessor, BlipForConditionalGeneration
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import pandas as pd

# Load T-shirt Dataset

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"saraa1z","key":"dd3e4cb2fe11bc403bcd5da736d17f5c"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d danizo/applied-data-mining-final-project-t-shirt-data

Dataset URL: https://www.kaggle.com/datasets/danizo/applied-data-mining-final-project-t-shirt-data
License(s): apache-2.0
 97% 348M/358M [00:01<00:00, 201MB/s]
100% 358M/358M [00:01<00:00, 206MB/s]


In [ ]:
!unzip applied-data-mining-final-project-t-shirt-data.zip -d T_shirt_dataset

Archive:  applied-data-mining-final-project-t-shirt-data.zip
  inflating: T_shirt_dataset/tshirt/1.jpg  
  inflating: T_shirt_dataset/tshirt/10.jpg  
  inflating: T_shirt_dataset/tshirt/100.jpg  
  inflating: T_shirt_dataset/tshirt/1000.jpg  
  inflating: T_shirt_dataset/tshirt/1001.jpg  
  inflating: T_shirt_dataset/tshirt/1002.jpg  
  inflating: T_shirt_dataset/tshirt/1003.jpg  
  inflating: T_shirt_dataset/tshirt/1004.jpg  
  inflating: T_shirt_dataset/tshirt/1005.jpg  
  inflating: T_shirt_dataset/tshirt/1006.jpg  
  inflating: T_shirt_dataset/tshirt/1007.jpg  
  inflating: T_shirt_dataset/tshirt/1008.jpg  
  inflating: T_shirt_dataset/tshirt/1009.jpg  
  inflating: T_shirt_dataset/tshirt/101.jpg  
  inflating: T_shirt_dataset/tshirt/1010.jpg  
  inflating: T_shirt_dataset/tshirt/1011.jpg  
  inflating: T_shirt_dataset/tshirt/1012.jpg  
  inflating: T_shirt_dataset/tshirt/1013.jpg  
  inflating: T_shirt_dataset/tshirt/1014.jpg  
  inflating: T_shirt_dataset/tshirt/1015.jpg  
  infl

In [ ]:
def load_dataset(image_folder):
    """
    Args:
        image_folder (str): Path to the folder containing T-shirt images.
    Returns:
        list: List of image file paths.
    """
    image_files = [os.path.join(image_folder, f) for f in os.listdir(image_folder) if f.endswith(('.jpg'))]
    return image_files

In [ ]:
from matplotlib import pyplot as plt
def visualize_top_images(top_image_paths):
    plt.figure(figsize=(15, 10))
    for i, image_path in enumerate(top_image_paths, 1):
        plt.subplot(1, 5, i)
        img = Image.open(image_path)
        plt.imshow(img)
        plt.title(f"Image {i}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
device = torch.device('cuda')
device

device(type='cuda')

# Initialize Models

## CLIP

CLIP’s architecture involves parallel training of a text transformer and a vision transformer (or ResNet), producing 512-dimensional embeddings that align similar concepts across modalities. Contrastive pretraining is central to CLIP’s success, training the model to maximize similarity between matching text-image pairs while minimizing it for mismatched pairs. This framework allows CLIP to generalize across diverse datasets, bridging the gap between natural language processing and computer vision.

CLIP actually consists of two models trained in parallel. A 12-layer text transformer for building text embeddings and a ResNet or vision transformer (ViT) for building image embeddings

![image.png](../assets/fig-02.png)

In the image below, we can see a single pretraining step on a single batch. The loss function assumes pairs in the diagonal should have a maximized dot product score, and all other pairs should have a minimized dot product score. Both text and image encoder models are optimized for this.

![image.png](../assets/fig-03.png)

Similar text and images will be encoded into a similar vector space. Dissimilar text and images do not share a similar vector space.

![image.png](../assets/fig-04.png)

In [ ]:
def initialize_clip():
    """
    Initialize the CLIP model and processor.
    Returns:
        model: Pretrained CLIP model.
        processor: CLIP processor for preprocessing inputs.
    """
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    return model, processor

## BLIP

Bootstrapping Language-
Image Pre-training for unified vision-language understand-
ing and generation. BLIP is a new VLP framework which
enables a wider range of downstream tasks than existing
methods. It introduces two contributions from the model
and data perspective, respectively:

(a) Multimodal mixture of Encoder-Decoder (MED): a new
model architecture for effective multi-task pre-training and
flexible transfer learning. An MED can operate either as
a unimodal encoder, or an image-grounded text encoder,
or an image-grounded text decoder. The model is jointly
pre-trained with three vision-language objectives: image-
text contrastive learning, image-text matching, and image-
conditioned language modeling.

(b) Captioning and Filtering (CapFilt): a new dataset boos-
trapping method for learning from noisy image-text pairs.
We finetune a pre-trained MED into two modules: a cap-
tioner to produce synthetic captions given web images, and
a filter to remove noisy captions from both the original web
texts and the synthetic texts.

The model uses Vision Transformer ViT (Dosovitskiy et al., 2021) which divides the input image into patches and encodes them as a sequence of embedding with addition to [CLS] token to represent the globe image feature. ViT uses less computation cost and is a straightforward method, and is being adopted by recent methods.

![bf9bd9a4-da80-4059-bdc1-3f49549d4044.png](../assets/fig-05.png)

In [ ]:
def initialize_blip():
    """
    Initialize the BLIP model and processor.
    Returns:
        model: Pretrained BLIP model.
        processor: BLIP processor for preprocessing inputs.
    """
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

    return model, processor

Why I used BLIP over MinicPM?

MinicPM is generally faster than BLIP due to its lightweight architecture and optimizations. It is designed for efficient computation and can be preferable for applications where speed is a critical requirement.
BLIP, while potentially slower, **provide higher-quality captioning** results due to its more complex model design. It may be a better choice when high accuracy and detailed output are prioritized over speed.

In [ ]:
# def initialize_minicpm():
#     """
#     Initialize the minicpm model and processor.
#     Returns:
#         model: Pretrained minicpm model.
#         processor: minicpm processor for preprocessing inputs.
#     """
#     # Load MiniCPM model and processor
#     processor = AutoProcessor.from_pretrained("openbmb/MiniCPM-V")
#     model = AutoModelForVision2Seq.from_pretrained("openbmb/MiniCPM-V")

#     return model, processor

## Sentence Transformer

Sentence Transformers (a.k.a. SBERT) is the go-to Python module for accessing, using, and training state-of-the-art text and image embedding models.

BERT uses an encoder-only transformer focused on sentence embedding, but S-BERT leverages a sentence-specific transformer known as a Siamese transformer. This type of transformer grasps the semantic meaning of sentences, making it ideal for tasks that require understanding semantic similarities.

all-MiniLM-L6-v2
This is a sentence-transformers model: It maps sentences & paragraphs to a 384 dimensional dense vector space and can be used for tasks like clustering or semantic search.

In [ ]:
def initialize_sentence_transformer():
    """
    Initialize the Sentence Transformers model.
    Returns:
        model: Pretrained Sentence Transformers model.
    """
    model = SentenceTransformer('all-MiniLM-L6-v2')
    return model

## laod data files

In [ ]:
image_folder = "/content/T_shirt_dataset/tshirt"
image_files = load_dataset(image_folder)
print(f"Loaded {len(image_files)} images from the dataset.")

Loaded 1993 images from the dataset.


# Image Retrivial

In [ ]:
def retrieve_images(query_text, image_files, clip_model, clip_processor, top_k=5):
    """
    Retrieve the most similar images for a given text query.
    Args:
        query_text (str): Text query (e.g., "blue graphic T-shirt").
        image_files (list): List of image file paths.
        clip_model: Pretrained CLIP model.
        clip_processor: CLIP processor.
        top_k (int): Number of similar images to retrieve.
    Returns:
        list: List of top-k relevant image file paths.
    """

    if torch.cuda.is_available():
        clip_model = clip_model.to("cuda")


    inputs_text = clip_processor(text=[query_text], return_tensors="pt", padding=True)
    if torch.cuda.is_available():
        inputs_text = {key: value.to("cuda") for key, value in inputs_text.items()}
    query_embedding = clip_model.get_text_features(**inputs_text)

    # Compute similarity with all images
    similarities = []
    for image_path in image_files:
        try:
            image = Image.open(image_path).convert("RGB")
            inputs_image = clip_processor(images=image, return_tensors="pt")
            if torch.cuda.is_available():
                inputs_image = {key: value.to("cuda") for key, value in inputs_image.items()}


            image_embedding = clip_model.get_image_features(**inputs_image)

            # Compute cosine similarity between query and image embeddings
            similarity = torch.nn.functional.cosine_similarity(query_embedding, image_embedding).item()
            similarities.append((image_path, similarity))

        except Exception as e:
            print(f"Error processing {image_path}: {e}")
            similarities.append((image_path, -1))


    similarities.sort(key=lambda x: x[1], reverse=True)
    return [x[0] for x in similarities[:top_k]]

In [ ]:
# Run the model
clip_model, clip_processor = initialize_clip()

# Example
query_text = "Pink striped T-shirt"
top_k_images = retrieve_images(query_text, image_files, clip_model, clip_processor, top_k=5)
print(f"Top {len(top_k_images)} similar images for query '{query_text}': {top_k_images}")
visualize_top_images(top_k_images)

# Image Captioning

In [ ]:
def generate_captions(image_files, blip_model, blip_processor, output_csv='captions.csv'):
    """
    Generate captions for all images in the dataset.
    Args:
        image_files (list): List of image file paths.
        blip_model: Pretrained BLIP model.
        blip_processor: BLIP processor.
        output_csv (str): Path to save the output CSV file.
    Returns:
        dict: Dictionary mapping image file paths to their captions.
    """
    # Move the model to CUDA if available
    if torch.cuda.is_available():
        blip_model = blip_model.to("cuda")

    captions = {}
    counter = 0

    for image_path in image_files:
        try:
            image = Image.open(image_path).convert("RGB")
            inputs = blip_processor(image, return_tensors="pt")
            if torch.cuda.is_available():
                inputs = {key: value.to("cuda") for key, value in inputs.items()}

            # Generate the caption
            with torch.no_grad():  # Disable gradient calculation for efficiency
                outputs = blip_model.generate(**inputs)

            # Decode the caption
            caption = blip_processor.decode(outputs[0], skip_special_tokens=True)
            captions[image_path] = caption

            counter += 1
            print(f"Processed image {counter}/{len(image_files)}: {image_path}")

        except Exception as e:
            print(f"Error processing {image_path}: {e}")
            captions[image_path] = "Caption generation failed"

    # Save captions to a CSV file to pass to text retrieval
    df = pd.DataFrame(list(captions.items()), columns=['Image_Path', 'Caption'])
    df.to_csv(output_csv, index=False)

    return captions

In [ ]:
blip_model, blip_processor = initialize_blip()
captions = generate_captions(image_files, blip_model, blip_processor)
print(f"Generated captions for {len(captions)} images.")

Processed image 1/1993: /content/T_shirt_dataset/tshirt/802.jpg
Processed image 2/1993: /content/T_shirt_dataset/tshirt/1189.jpg
Processed image 3/1993: /content/T_shirt_dataset/tshirt/251.jpg
Processed image 4/1993: /content/T_shirt_dataset/tshirt/1803.jpg
Processed image 5/1993: /content/T_shirt_dataset/tshirt/1651.jpg
Processed image 6/1993: /content/T_shirt_dataset/tshirt/874.jpg
Processed image 7/1993: /content/T_shirt_dataset/tshirt/1359.jpg
Processed image 8/1993: /content/T_shirt_dataset/tshirt/982.jpg
Processed image 9/1993: /content/T_shirt_dataset/tshirt/15.jpg
Processed image 10/1993: /content/T_shirt_dataset/tshirt/711.jpg
Processed image 11/1993: /content/T_shirt_dataset/tshirt/132.jpg
Processed image 12/1993: /content/T_shirt_dataset/tshirt/1699.jpg
Processed image 13/1993: /content/T_shirt_dataset/tshirt/1725.jpg
Processed image 14/1993: /content/T_shirt_dataset/tshirt/1180.jpg
Processed image 15/1993: /content/T_shirt_dataset/tshirt/354.jpg
Processed image 16/1993: /co

# Text Retrieval

In [ ]:
def retrieve_images_by_text(query_text, captions, sentence_model, top_k=5):
    """
    Retrieve the most relevant images for a given text query based on captions.
    Args:
        query_text (str): Text query (e.g., "blue graphic T-shirt").
        captions (dict): Dictionary mapping image file paths to their captions.
        sentence_model: Pretrained Sentence Transformers model.
        top_k (int): Number of relevant images to retrieve.
    Returns:
        list: List of top-k relevant image file paths.
    """
    # Extract query text embedding
    query_embedding = sentence_model.encode(query_text)

    # Compute similarity with all captions
    similarities = []
    for image_path, caption in captions.items():
        caption_embedding = sentence_model.encode(caption)
        similarity = cosine_similarity([query_embedding], [caption_embedding])[0][0]
        similarities.append((image_path, similarity))

    # Sort by similarity and return top-k
    similarities.sort(key=lambda x: x[1], reverse=True)
    return [x[0] for x in similarities[:top_k]]


In [ ]:
def load_captions_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    captions = dict(zip(df['Image_Path'], df['Caption']))
    return captions

In [ ]:
sentence_model = initialize_sentence_transformer()
csv_path = "captions.csv"
captions = load_captions_from_csv(csv_path)

# Text Retrieval
query_text = "Pink striped T-shirt"
top_k_relevant_images = retrieve_images_by_text(query_text, captions, sentence_model, top_k=5)
print(f"Top {len(top_k_relevant_images)} relevant images for query '{query_text}': {top_k_relevant_images}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Top 5 relevant images for query 'Pink striped T-shirt': ['/content/T_shirt_dataset/tshirt/1247.jpg', '/content/T_shirt_dataset/tshirt/1971.jpg', '/content/T_shirt_dataset/tshirt/1539.jpg', '/content/T_shirt_dataset/tshirt/276.jpg', '/content/T_shirt_dataset/tshirt/703.jpg']


In [ ]:
visualize_top_images(top_k_relevant_images)

# Test as a Product

In [ ]:
def main(query_text , k):
    # Path to the dataset
    image_folder = "/content/T_shirt_dataset/tshirt"

    # Load the dataset
    image_files = load_dataset(image_folder)
    print(f"Loaded {len(image_files)} images from the dataset.")

    # Initialize models
    clip_model, clip_processor = initialize_clip()
    blip_model, blip_processor = initialize_blip()
    sentence_model = initialize_sentence_transformer()

    # Image Retrieval
    top_k_images = retrieve_images(query_text, image_files, clip_model, clip_processor, top_k= k)
    print(f"Top {len(top_k_images)} similar images for query '{query_text}': {top_k_images}")

    # Image Captioning
    # captions = generate_captions(image_files, blip_model, blip_processor)
    # print(f"Generated captions for {len(captions)} images.")
    csv_path = "captions.csv"
    captions = load_captions_from_csv(csv_path)

    # Text Retrieval
    top_k_relevant_images_by_text = retrieve_images_by_text(query_text, captions, sentence_model, top_k= k)
    print(f"Top {len(top_k_relevant_images_by_text)} relevant images for query '{query_text}': {top_k_relevant_images_by_text}")

    return top_k_images , top_k_relevant_images_by_text

if __name__ == "__main__":
    query = "blue graphic T-shirt"
    top_k = 5
    top_k_images , top_k_relevant_images = main(query_text = query , k = top_k)

Loaded 1993 images from the dataset.


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Top 5 similar images for query 'blue graphic T-shirt': ['/content/T_shirt_dataset/tshirt/1597.jpg', '/content/T_shirt_dataset/tshirt/1226.jpg', '/content/T_shirt_dataset/tshirt/1839.jpg', '/content/T_shirt_dataset/tshirt/1844.jpg', '/content/T_shirt_dataset/tshirt/1375.jpg']
Top 5 relevant images for query 'blue graphic T-shirt': ['/content/T_shirt_dataset/tshirt/1514.jpg', '/content/T_shirt_dataset/tshirt/1499.jpg', '/content/T_shirt_dataset/tshirt/1579.jpg', '/content/T_shirt_dataset/tshirt/1884.jpg', '/content/T_shirt_dataset/tshirt/1084.jpg']


In [ ]:
visualize_top_images(top_k_images)

In [ ]:
visualize_top_images(top_k_relevant_images)

# additional

these are the test results while I was testing on 199 data for the same query.

In [ ]:
blip_model, blip_processor = initialize_blip()
image_folder = "/content/T_shirt_dataset/tshirt"
image_files = [os.path.join(image_folder, f) for f in os.listdir(image_folder) if f.endswith(('1031.jpg', '.png'))]
captions = generate_captions(image_files, blip_model, blip_processor)
print(captions)

{'/content/T_shirt_dataset/tshirt/1031.jpg': 'the elder tee - shirt in blue'}


In [ ]:
visualize_top_images(['/content/T_shirt_dataset/tshirt/1031.jpg', '/content/T_shirt_dataset/tshirt/1071.jpg', '/content/T_shirt_dataset/tshirt/191.jpg', '/content/T_shirt_dataset/tshirt/1661.jpg', '/content/T_shirt_dataset/tshirt/931.jpg'])

In [ ]:
visualize_top_images(['/content/T_shirt_dataset/tshirt/1671.jpg', '/content/T_shirt_dataset/tshirt/41.jpg', '/content/T_shirt_dataset/tshirt/91.jpg', '/content/T_shirt_dataset/tshirt/931.jpg', '/content/T_shirt_dataset/tshirt/1031.jpg'])